# OpenAdapt End-to-End Training Notebook

This notebook is the submission rerun path for TRL + Unsloth + OpenEnv.

## 1) GPU initialization

In [ ]:
!nvidia-smi
import torch
print('cuda:', torch.cuda.is_available())
if torch.cuda.is_available():
    print('gpu:', torch.cuda.get_device_name(0))
    print('bf16_supported:', torch.cuda.is_bf16_supported())

## 2) Install dependencies (Unsloth + TRL + OpenEnv)

In [ ]:
%cd /content
!rm -rf adaptive-meta-env
!git clone https://github.com/rithunkp/adaptive-meta-env.git
%cd /content/adaptive-meta-env
!python -m pip install -U pip setuptools wheel
!pip install trl transformers datasets accelerate peft bitsandbytes wandb
!pip install unsloth
!pip install git+https://github.com/meta-pytorch/OpenEnv.git

## 3) Preflight validate/test

In [ ]:
%cd /content/adaptive-meta-env
!python -m unittest discover -s tests -p "test_openenv_incident_triage.py" -v
!cd envs/incident_triage && openenv validate --verbose

## 4) Configure tokens/secrets

In [ ]:
import os
from google.colab import userdata
os.environ['HF_TOKEN'] = userdata.get('HF_TOKEN')
os.environ['WANDB_API_KEY'] = userdata.get('WANDB_API_KEY')
print('HF_TOKEN set:', bool(os.environ.get('HF_TOKEN')))
print('WANDB_API_KEY set:', bool(os.environ.get('WANDB_API_KEY')))

## 5) Smoke train

In [ ]:
%cd /content/adaptive-meta-env
!python hf_space_trainer.py \
  --model-name meta-llama/Llama-3.1-8B-Instruct \
  --max-steps 20 \
  --learning-rate 1e-5 \
  --output-dir artifacts/grpo_smoke \
  --final-model-dir artifacts/final_smoke \
  --run-name openadapt-smoke \
  --env-path generated_envs/incident_triage_env.py \
  --env-class IncidentTriageEnv

## 6) Real train run

In [ ]:
%cd /content/adaptive-meta-env
!python hf_space_trainer.py \
  --model-name meta-llama/Llama-3.1-8B-Instruct \
  --max-steps 1200 \
  --learning-rate 1e-5 \
  --output-dir artifacts/grpo_main \
  --final-model-dir artifacts/final_main \
  --run-name openadapt-main \
  --env-path generated_envs/incident_triage_env.py \
  --env-class IncidentTriageEnv

## 7) Export metrics + plots and quick baseline vs trained summary

In [ ]:
%cd /content/adaptive-meta-env
!python -m src.training.train_grpo --episodes 40 --output artifacts/training_metrics.jsonl
!python -m src.training.export_submission_artifacts
!ls -lah artifacts/plots
!cat artifacts/metrics/submission_summary.json